# Generating the Knowledge Distillation Dataset — `all_mixed` (Full-Range Variant)

This notebook generates a **highly diverse `all_mixed` dataset** for training the teacher model
(RouteNet-Fermi) used in QoS-aware SDN routing via Knowledge Distillation.

## Design difference from the original split
The original `all_mixed` spec trains **only on 10-node topologies** and tests on larger sizes,
specifically to evaluate zero-shot generalisation. This notebook instead covers **10 → 300 nodes
across all three splits** (train / validation / test) so the teacher model gains direct exposure
to the full network-scale range before distilling knowledge to the lightweight student GNN.

| Split      | Node sizes                                          | Topos/size | Sched configs | Routings/topo |
|------------|-----------------------------------------------------|-----------|---------------|---------------|
| Train      | 10, 50, 75, 100, 130, 170, 200, 240, 260, 280, 300 | 7         | 50            | 10            |
| Test       | 10, 50, 75, 100, 130, 170, 200, 240, 260, 280, 300 | 2         | 5             | 10            |
| Validation | 10, 50, 75, 100, 130, 170, 200, 240, 260, 280, 300 | 1         | 5             | 10            |

For the original spec see [input_parameters_glossary.ipynb](input_parameters_glossary.ipynb).


In [1]:
import networkx as nx
import random
import os
import numpy as np

random.seed(42)
np.random.seed(42)

# ============================================================
#  GLOBAL CONFIGURATION  — edit only this block
# ============================================================

BASE_DATASET_PATH = "../data/sim_data_v1"

# Node sizes present in every split
# ALL_SIZES = [10, 50, 75, 100, 130, 170, 200, 240, 260, 280, 300]
ALL_SIZES = [10]


SPLITS = {
    "train": {
        "sizes":               ALL_SIZES,
        "topologies_per_size": 1,    # 7 random topologies per size
        "routings_per_topo":   1,   # 10 shortest-path variations
        "sched_configs":       1,   # 50 scheduling configurations
    },
    "test": {
        "sizes":               ALL_SIZES,
        "topologies_per_size": 1,
        "routings_per_topo":   1,
        "sched_configs":       1,
    },
    "validation": {
        "sizes":               ALL_SIZES,
        "topologies_per_size": 1,
        "routings_per_topo":   1,
        "sched_configs":       1,
    },
}

# ── Packet-size distributions (exactly 5 as per spec) ──
PKT_DIST_PROFILES = [
    "0,300,0.2,500,0.2,800,0.2,1000,0.2,1500,0.2",    # 1 – uniform spread
    "0,100,0.1,400,0.3,700,0.2,1200,0.3,1400,0.1",    # 2 – mid-heavy
    "0,200,0.4,600,0.15,900,0.15,1300,0.1,1450,0.2",  # 3 – small-packet heavy
    "0,64,0.5,256,0.2,512,0.1,1024,0.1,1500,0.1",     # 4 – VoIP / tiny packets
    "0,500,0.1,750,0.1,1000,0.3,1250,0.2,1500,0.3",   # 5 – large-packet / video
]

# Time-distribution codes: Poisson=0, CBR=1, ON-OFF=2,<on>,<off>
TIME_DIST_OPTIONS = ["0", "1", "2,10,5"]

# Ensure base directory exists
os.makedirs(BASE_DATASET_PATH, exist_ok=True)
print(f"Base path ready: {BASE_DATASET_PATH}")
print(f"Splits defined : {list(SPLITS.keys())}")
print(f"Node sizes     : {ALL_SIZES}")


Base path ready: ../data/sim_data_v1
Splits defined : ['train', 'test', 'validation']
Node sizes     : [10]


## Step 1 — Topology Structure Generation

Generates the **graph skeleton** (nodes + edges) without assigning scheduling policies.
Scheduling is applied separately so we can produce N independent scheduling configurations
on the same underlying graph.

Three topology models are used to ensure structural diversity:
- **Barabási-Albert** — scale-free, hub-and-spoke
- **Erdős-Rényi** — random, with connectivity guarantee (max 10 retries → BA fallback)
- **Watts-Strogatz** — small-world


In [2]:
def generate_graph_structure(net_size):
    """Return a NetworkX Graph with nodes/edges but NO scheduling attributes."""
    topo_type = random.choice(["barabasi", "erdos", "watts"])

    if topo_type == "barabasi":
        G = nx.barabasi_albert_graph(net_size, random.choice([2, 3]))

    elif topo_type == "erdos":
        G = nx.erdos_renyi_graph(net_size, p=random.uniform(0.05, 0.15))
        max_retries, retries = 10, 0
        while not nx.is_connected(G):
            if retries >= max_retries:
                print(f"  [WARN] Erdős-Rényi failed to connect after {max_retries} "
                      f"tries (net_size={net_size}). Falling back to Barabási-Albert.")
                G = nx.barabasi_albert_graph(net_size, random.choice([2, 3]))
                break
            G = nx.erdos_renyi_graph(net_size, p=random.uniform(0.05, 0.15))
            retries += 1

    else:  # watts-strogatz
        G = nx.watts_strogatz_graph(net_size, k=4, p=0.3)

    # Attach edge attributes (bandwidth & delay) — fixed per topology
    H = nx.Graph()
    H.graph["levelsToS"] = 3
    for n in G.nodes():
        H.add_node(n)
    for u, v in G.edges():
        bw    = random.choice([1e4, 5e4, 1e5, 1e6])   # bps
        delay = round(random.uniform(1, 20), 2)        # ms
        H.add_edge(u, v, bandwidth=bw, delay=delay)

    return H


## Step 2 — Apply Scheduling Configuration

Assigns a **fresh random scheduling policy** (FIFO / SP / WFQ / DRR) and buffer size to
every node in the graph, then writes a GML file.  Calling this N times on the same graph
produces N distinct scheduling configurations — matching the spec requirement of 50 configs
for training and 5 for test/validation.


In [3]:
def apply_scheduling_and_save(G, graph_file):
    """
    Assign random scheduling policy + buffer size to every node and write GML.
    Returns the annotated graph (edge attrs are preserved, node attrs replaced).
    """
    H = G.copy()
    H.graph["levelsToS"] = 3

    for n in H.nodes():
        policy = random.choice(["FIFO", "SP", "WFQ", "DRR"])
        H.nodes[n]["schedulingPolicy"] = policy
        H.nodes[n]["bufferSizes"]      = random.choice([8000, 16000, 32000, 64000])

        if policy == "FIFO":
            H.nodes[n]["tosToQoSqueue"] = "0,1,2"
        else:
            H.nodes[n]["tosToQoSqueue"] = "0;1;2"
            if policy in ["WFQ", "DRR"]:
                # Normalised integer weights that sum close to 100
                raw     = np.random.dirichlet(np.ones(3))
                weights = [max(1, int(round(w * 100))) for w in raw]
                # Adjust last weight so sum == 100
                weights[2] = max(1, 100 - weights[0] - weights[1])
                H.nodes[n]["schedulingWeights"] = (
                    f"{weights[0]},{weights[1]},{weights[2]}"
                )

    nx.write_gml(H, graph_file)
    return H


## Step 3 — Routing Generation

Generates **10 shortest-path variations** per topology by randomising edge weights before
running all-pairs Dijkstra.  Three weight strategies are used:
- `random`    — uniform random weights
- `delay`     — weight = propagation delay
- `bandwidth` — weight = 1/bandwidth (prefer high-BW links)


In [4]:
def generate_routing(G, routing_file):
    """Compute all-pairs shortest paths with randomised weights and write to file."""
    routing_type = random.choice(["random", "delay", "bandwidth"])

    for u, v in G.edges():
        if routing_type == "random":
            G[u][v]["weight"] = random.uniform(1, 10)
        elif routing_type == "delay":
            G[u][v]["weight"] = G[u][v]["delay"]
        else:
            G[u][v]["weight"] = 1.0 / G[u][v]["bandwidth"]

    paths = dict(nx.all_pairs_dijkstra_path(G, weight="weight"))

    with open(routing_file, "w") as f:
        for src in G.nodes():
            for dst in G.nodes():
                if src == dst:
                    continue
                path = ",".join(map(str, paths[src][dst]))
                f.write(path + "\n")

    # Remove temporary weight attribute
    for u, v in G.edges():
        G[u][v].pop("weight", None)


## Step 4 — Traffic Matrix Generation

Follows the spec exactly:
- `maxAvgLbda` drawn uniformly between **400 and 2000 bps** per sample
- Per-path bandwidth = `maxAvgLbda × Uniform(0.1, 1.0)`
- Time distribution: Poisson / CBR / ON-OFF — selected randomly per path
- Packet-size distribution: one of 5 fixed profiles selected per path
- ToS: 0, 1, or 2 — selected randomly per path
- Flow sparsity (~20%) with hot-node boost to prevent O(N²) explosion


In [5]:
def generate_tm(G, max_avg_lbda, traffic_file, sparsity=0.2):
    """
    Generate a traffic matrix file.

    bandwidth per path = max_avg_lbda * Uniform(0.1, 1.0)   [as per spec]
    """
    nodes    = list(G.nodes())
    num_hot  = max(1, int(0.1 * len(nodes)))
    hot_nodes = set(random.sample(nodes, num_hot))

    with open(traffic_file, "w") as f:
        for src in nodes:
            for dst in nodes:
                if src == dst:
                    continue

                # Hotspot-aware sparsity
                prob = sparsity * (2 if (src in hot_nodes or dst in hot_nodes) else 1)
                if random.random() > prob:
                    continue

                # Bandwidth per path — spec: maxAvgLbda × Uniform(0.1, 1.0)
                avg_bw = int(max_avg_lbda * random.uniform(0.1, 1.0))
                avg_bw = max(100, avg_bw)

                td  = random.choice(TIME_DIST_OPTIONS)
                sd  = random.choice(PKT_DIST_PROFILES)
                tos = random.choice([0, 1, 2])

                f.write(f"{src},{dst},{avg_bw},{td},{sd},{tos}\n")


## Step 5 — Main Generation Loop

Iterates over all splits, sizes, topologies, scheduling configurations, routings and
traffic matrices according to the config defined in Cell 1.

**Directory structure produced:**
```
sim_data_v1/
  train/
    scheduling-topo-10-1/
      graphs/      graph_10_1_sched0.txt  …
      routings/    routing_10_1_sched0_r0.txt  …
      tm/          tm_10_1_sched0_r0_tm0.txt   …
      simulation.txt
  test/
    …
  validation/
    …
```


In [6]:
total_samples = 0

for split_name, cfg in SPLITS.items():

    split_path = os.path.join(BASE_DATASET_PATH, split_name)
    os.makedirs(split_path, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  SPLIT: {split_name.upper()}")
    print(f"{'='*60}")

    for net_size in cfg["sizes"]:
        for topo_id in range(1, cfg["topologies_per_size"] + 1):

            dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
            dataset_path = os.path.join(split_path, dataset_name)

            graphs_dir   = os.path.join(dataset_path, "graphs")
            routings_dir = os.path.join(dataset_path, "routings")
            tm_dir       = os.path.join(dataset_path, "tm")
            os.makedirs(graphs_dir,   exist_ok=True)
            os.makedirs(routings_dir, exist_ok=True)
            os.makedirs(tm_dir,       exist_ok=True)

            print(f"  Generating {split_name}/{dataset_name} ...")

            # ── 1. Generate base graph structure (once per topology) ──────────
            G_base = generate_graph_structure(net_size)

            sim_lines = []

            # ── 2. Loop over scheduling configurations ────────────────────────
            for sched_id in range(cfg["sched_configs"]):

                graph_filename = f"graph_{net_size}_{topo_id}_sched{sched_id}.txt"
                graph_path     = os.path.join(graphs_dir, graph_filename)

                # Apply fresh random scheduling to the same base graph
                G_sched = apply_scheduling_and_save(G_base, graph_path)

                # ── 3. Loop over routing variations ───────────────────────────
                for r_id in range(cfg["routings_per_topo"]):

                    routing_filename = (
                        f"routing_{net_size}_{topo_id}_sched{sched_id}_r{r_id}.txt"
                    )
                    routing_path = os.path.join(routings_dir, routing_filename)
                    generate_routing(G_sched, routing_path)

                    # ── 4. Generate one traffic matrix per routing ────────────
                    max_avg_lbda = random.randint(400, 2000)
                    tm_filename  = (
                        f"tm_{net_size}_{topo_id}_sched{sched_id}_r{r_id}.txt"
                    )
                    tm_path = os.path.join(tm_dir, tm_filename)
                    generate_tm(G_sched, max_avg_lbda, tm_path)

                    sim_lines.append(
                        f"graphs/{graph_filename},"
                        f"routings/{routing_filename},"
                        f"tm/{tm_filename}\n"
                    )
                    total_samples += 1

            # Write simulation index file
            sim_file = os.path.join(dataset_path, "simulation.txt")
            with open(sim_file, "w") as fd:
                fd.writelines(sim_lines)

print(f"\n Dataset generation complete — {total_samples:,} samples total.")
print("Ready for Docker simulation.")



  SPLIT: TRAIN
  Generating train/scheduling-topo-10-1 ...

  SPLIT: TEST
  Generating test/scheduling-topo-10-1 ...

  SPLIT: VALIDATION
  Generating validation/scheduling-topo-10-1 ...
  [WARN] Erdős-Rényi failed to connect after 10 tries (net_size=10). Falling back to Barabási-Albert.

 Dataset generation complete — 3 samples total.
Ready for Docker simulation.


## Step 6 — Generate `conf.yml` for Every Dataset Folder

One configuration file per dataset directory is required by BNNetSimulator.
The `threads` and `samples_per_file` values can be tuned based on your hardware.


In [7]:
import yaml

for split_name, cfg in SPLITS.items():
    split_path = os.path.join(BASE_DATASET_PATH, split_name)

    for net_size in cfg["sizes"]:
        for topo_id in range(1, cfg["topologies_per_size"] + 1):

            dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
            dataset_path = os.path.join(split_path, dataset_name)

            if not os.path.exists(dataset_path):
                continue

            conf = {
                "threads":          6,
                "dataset_name":     dataset_name,
                "samples_per_file": 10,
                "rm_prev_results":  "n",
                "write_pkt_info":   "n",
            }

            with open(os.path.join(dataset_path, "conf.yml"), "w") as fd:
                yaml.dump(conf, fd)

print("conf.yml files generated for all dataset folders.")


conf.yml files generated for all dataset folders.


## Step 7 — Run BNNetSimulator (Sequential Docker Execution)

Runs the simulator for every dataset folder one at a time to avoid memory exhaustion,
especially important for large (200–300 node) topologies.

Each `out.log` in the dataset folder contains one line per sample.  
A status of **`Ok`** confirms the sample completed successfully.


In [8]:
from getpass import getpass
import subprocess

use_sudo = os.name != "nt"
pwd = ""

if use_sudo:
    print("Superuser privileges required for Docker. Enter sudo password:")
    pwd = getpass()

print("Starting batch Docker simulations...\n")

errors = []

for split_name, cfg in SPLITS.items():
    split_path = os.path.join(BASE_DATASET_PATH, split_name)

    for net_size in cfg["sizes"]:
        for topo_id in range(1, cfg["topologies_per_size"] + 1):

            dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
            dataset_path = os.path.join(split_path, dataset_name)

            if not os.path.exists(dataset_path):
                print(f"  [SKIP] {split_name}/{dataset_name} — directory missing.")
                continue

            abs_path = os.path.abspath(dataset_path)
            print(f"---> [{split_name}] Simulating {dataset_name} ...")

            raw_cmd = (
                f"docker run --rm "
                f"--mount type=bind,src={abs_path},dst=/data "
                f"bnnupc/bnnetsimulator"
            )
            cmd = f"echo {pwd} | sudo -S {raw_cmd}" if use_sudo else raw_cmd

            try:
                subprocess.run(cmd, shell=True, check=True)
                print(f"     Done: {dataset_name}")
            except subprocess.CalledProcessError as e:
                msg = f"[ERROR] {split_name}/{dataset_name}: {e}"
                print(f"     {msg}")
                errors.append(msg)

print("\n" + "="*60)
if errors:
    print(f"Completed with {len(errors)} error(s):")
    for e in errors:
        print(f"  {e}")
else:
    print("All simulations completed successfully!")


Superuser privileges required for Docker. Enter sudo password:
Starting batch Docker simulations...

---> [train] Simulating scheduling-topo-10-1 ...


[sudo] password for abaragithan: INFO:root:0: OK


     Done: scheduling-topo-10-1
---> [test] Simulating scheduling-topo-10-1 ...


[sudo] password for abaragithan: INFO:root:0: OK


     Done: scheduling-topo-10-1
---> [validation] Simulating scheduling-topo-10-1 ...


[sudo] password for abaragithan: INFO:root:0: OK


     Done: scheduling-topo-10-1

All simulations completed successfully!


## Step 8 — Monitor Simulation Quality

Scans every `out.log` file and counts `Ok` vs failed samples.
Run this after Docker simulation completes to verify data quality before training.


In [11]:
summary = {}

for split_name, cfg in SPLITS.items():
    split_path = os.path.join(BASE_DATASET_PATH, split_name)
    ok_total, fail_total = 0, 0

    for net_size in cfg["sizes"]:
        for topo_id in range(1, cfg["topologies_per_size"] + 1):

            dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
            log_file     = os.path.join(split_path, dataset_name, "out.log")

            if not os.path.exists(log_file):
                continue

            with open(log_file) as f:
                lines = f.readlines()

            ok   = sum(1 for l in lines if "Ok" in l)
            fail = len(lines) - ok
            ok_total   += ok
            fail_total += fail

    summary[split_name] = {"ok": ok_total, "failed": fail_total}

print("\n📊 Simulation Quality Report")
print("-" * 40)
for split, counts in summary.items():
    total = counts["ok"] + counts["failed"]
    pct   = (counts["ok"] / total * 100) if total > 0 else 0
    print(f"  {split:12s}: {counts['ok']:>6,} Ok  |  {counts['failed']:>4,} failed  "
          f"({pct:.1f}% success)")
print("-" * 40)



📊 Simulation Quality Report
----------------------------------------
  train       :      0 Ok  |     1 failed  (0.0% success)
  test        :      0 Ok  |     1 failed  (0.0% success)
  validation  :      0 Ok  |     1 failed  (0.0% success)
----------------------------------------
